## Natural Scenes Dataset Voxel Searchlight RDM Correlation

**setup**
- download the Natural Scenes Dataset (500 gigabytes !) [try this **helper](https://github.com/lucas-nunn/visuo_llm_ram_rescue/blob/main/src/nsd_visuo_semantics/utils/download_nsd_visuo_semantics.py)
- set up your [environment variables](../.env.example)
- `cp .env.example .env`
- choose a model, which may entail writing custom code for extracting embeddings and generating RDMs

**background**
- read this [paper](https://www.nature.com/articles/s42256-025-01072-0)
- go through this [repo](https://github.com/lucas-nunn/visuo_llm_ram_rescue)
- this notebook essentially runs 

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
env_path = next((candidate for candidate in env_candidates if candidate.exists()), None)
if env_path is None:
    raise FileNotFoundError("Missing .env file. Copy .env.example to .env and fill paths.")
load_dotenv(env_path)

MODEL = "all-mpnet-base-v2"
SUBJECT = 2
NUM_SESSIONS = 20

nsd_dir = os.environ["NSD_DATA_PATH"]
assert Path(nsd_dir).exists(), f"NSD_DATA_PATH does not exist: {nsd_dir}"
figures_dir = f"../figures"
save_path = "../results/embeddings/"
models_rdm_dist = "correlation"
base_save_dir = "../results/searchlight"
betas_dir = f"{base_save_dir}/betas"
precompsl_dir = f"{base_save_dir}/searchlights"
saved_embeddings_dir = f"{base_save_dir}/embeddings"
rdms_dir = f'{base_save_dir}/serialised_models_{models_rdm_dist}'

In [ ]:
from nsd_visuo_semantics.get_embeddings.get_nsd_sentence_embeddings_simple import get_nsd_sentence_embeddings_simple

get_nsd_sentence_embeddings_simple(MODEL, "../data/ms_coco_nsd_captions_test.pkl", "eh", save_path, True)

In [ ]:
from nsd_visuo_semantics.utils.nsd_prepare_modelrdms import nsd_prepare_modelrdms

nsd_prepare_modelrdms(MODEL, models_rdm_dist, saved_embeddings_dir, rdms_dir, nsd_dir, "", "", True, n_sessions=NUM_SESSIONS, n_subjects=SUBJECT)

In [ ]:
from nsd_visuo_semantics.searchlight_analyses.nsd_searchlight_main_tf import nsd_searchlight_main_tf

nsd_searchlight_main_tf(MODEL, models_rdm_dist, 
                        nsd_dir, base_save_dir, betas_dir, base_save_dir, 
                        False, subject=SUBJECT, n_sessions=NUM_SESSIONS)

In [ ]:
from nsd_visuo_semantics.searchlight_analyses.nsd_project_fsaverage import nsd_project_fsaverage

nsd_project_fsaverage([MODEL], models_rdm_dist, nsd_dir, base_save_dir)

In [ ]:
import cortex
from pathlib import Path

filestore = Path(cortex.db.filestore)
if not (filestore / "fsaverage" / "surfaces" / "wm_lh.gii").exists():
    raise RuntimeError(
        "Pycortex fsaverage surfaces not found. "
        "Run once: python scripts/setup_pycortex.py"
    )

In [ ]:
from nsd_visuo_semantics.utils.py_plot_brain_utils import pyplot_brains_from_models_list

pyplot_brains_from_models_list(
    [MODEL],
    [MODEL],
    f"{base_save_dir}/searchlight_respectedsampling_correlation",
    layer="last",
    contrast_layer="same",
    contrast_same_model=False,
    save_type="png",
    figpath=figures_dir,
    plot_indiv_sub=True,
    plot_subj_avg=False,
    roi_overlay="streams",
    nsd_dir=nsd_dir,
    roi_linecolor="black",
    roi_linewidth=0.8,
)